In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andrewmvd/road-sign-detection")

print("Path to dataset files:", path)

100%|██████████| 218M/218M [00:01<00:00, 138MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/andrewmvd/road-sign-detection/versions/1


In [4]:
!ls $path

annotations  images


In [7]:
import os
import io
import shutil
import requests
import matplotlib.pyplot as plt
import numpy as np
from glob import glob
from PIL import Image
from lxml import etree
from tqdm import tqdm
from pathlib import Path
import xml.etree.ElementTree as ET
import random

In [3]:
!pip install ultralytics
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [8]:
SEED = 42
class_dict = {"stop": 0, "speedlimit": 1, "crosswalk": 2, "trafficlight": 3}
SRC = Path(path)
OUT = Path("/content/yolo_data")

def voc_to_yolo(xml_path):
    root = ET.parse(xml_path).getroot()
    w = int(root.find("size/width").text)
    h = int(root.find("size/height").text)
    lines = []
    for obj in root.findall("object"):
        cls = class_dict[obj.find("name").text.strip()]
        b = obj.find("bndbox")
        x1, y1 = float(b.find("xmin").text), float(b.find("ymin").text)
        x2, y2 = float(b.find("xmax").text), float(b.find("ymax").text)
        lines.append(f"{cls} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
    return lines

def split_and_convert(train_size=0.7, val_size=0.15, seed=SEED):
    stems = sorted(p.stem for p in (SRC / "annotations").glob("*.xml"))
    random.Random(seed).shuffle(stems)
    n = len(stems)
    t, v = int(n * train_size), int(n * (train_size + val_size))
    splits = {"train": stems[:t], "val": stems[t:v], "test": stems[v:]}
    for split, names in splits.items():
        (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)
        for s in names:
            shutil.copy(SRC / "images" / f"{s}.png", OUT / "images" / split / f"{s}.png")
            (OUT / "labels" / split / f"{s}.txt").write_text("\n".join(voc_to_yolo(SRC / "annotations" / f"{s}.xml")))
        print(split, len(names), "images")

split_and_convert()

train 613 images
val 132 images
test 132 images
